# 🦎 Galápagos YOLO — Anotación y Entrenamiento

Lee imágenes desde el **banco compartido** (`galapagos_wildlife/raw_images/`).
Para descargar o agregar más imágenes usa `00_download_dataset.ipynb`.

**Fases:**
1. Setup (lee banco compartido)
2. Auto-anotación YOLO World (~2-3h)
3. Entrenamiento YOLOv8n (~2-3h)
4. Export TFLite + Descarga

**Salida:** `MyDrive/galapagos_wildlife/yolo/models/`


## Celda 0 — Montar Drive (RECOMENDADO para no perder datos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/galapagos_yolo'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive montado. Checkpoints y modelo final en: {DRIVE_DIR}')

## Recuperación — Ejecutar SOLO si el runtime se resetó

Si perdiste el runtime a mitad del proceso, ejecuta la celda de abajo según en qué fase estabas:
- **Reseteo durante Celda 3 (anotación):** restaura el dataset anotado desde Drive
- **Reseteo durante Celda 4 (entrenamiento):** restaura el dataset + retoma desde último checkpoint

Luego continúa desde la celda donde te quedaste.

In [ ]:
# Ejecutar Celda 0 y Celda 1 primero, luego esta celda

from pathlib import Path
import shutil, os

drive_dataset = Path(f'{DRIVE_DIR}/dataset')
local_dataset = Path('/content/dataset')

# ── Paso 1: Restaurar dataset anotado de Drive ──────────────────────────────
restored_files = 0
if drive_dataset.exists():
    for split in ['train', 'val']:
        for subdir in ['images', 'labels']:
            src = drive_dataset / subdir / split
            dst = local_dataset / subdir / split
            if src.exists():
                dst.mkdir(parents=True, exist_ok=True)
                for f in src.iterdir():
                    target = dst / f.name
                    if not target.exists():
                        shutil.copy(f, target)
                        restored_files += 1
    yaml_src = Path(f'{DRIVE_DIR}/galapagos.yaml')
    if yaml_src.exists():
        shutil.copy(yaml_src, '/content/dataset/galapagos.yaml')
    id_map_src = Path(f'{DRIVE_DIR}/class_id_map.json')
    if id_map_src.exists():
        shutil.copy(id_map_src, '/content/class_id_map.json')
    train_n = len(list((local_dataset / 'images' / 'train').glob('*.jpg')))
    val_n   = len(list((local_dataset / 'images' / 'val').glob('*.jpg')))
    print(f'Dataset restaurado: {train_n} train, {val_n} val (+{restored_files} archivos nuevos)')
else:
    print('No se encontro dataset anotado en Drive — ejecuta Celda 3 completa')

# ── Paso 2 (solo si reseteo durante entrenamiento): restaurar checkpoint ────
last_pt_drive = Path(f'{DRIVE_DIR}/last.pt')
best_pt_drive = Path(f'{DRIVE_DIR}/best.pt')

run_dir = Path('/content/runs/galapagos_yolo/weights')
run_dir.mkdir(parents=True, exist_ok=True)

if last_pt_drive.exists():
    shutil.copy(last_pt_drive, run_dir / 'last.pt')
    shutil.copy(last_pt_drive, run_dir / 'best.pt')  # por si acaso
    print(f'Checkpoint restaurado: last.pt desde Drive')
    print('Para retomar entrenamiento usa: YOLO("last.pt").train(resume=True)')
elif best_pt_drive.exists():
    shutil.copy(best_pt_drive, run_dir / 'best.pt')
    print(f'Solo best.pt disponible — no se puede hacer resume exacto')
    print('El entrenamiento arrancara desde best.pt (sin estado del optimizador)')
else:
    print('Sin checkpoints en Drive — ejecuta Celda 4 desde cero')


## Celda 1 — Setup

In [ ]:
!pip install ultralytics -q

import os, json, time, shutil, random, gc
import urllib.request, urllib.parse
from datetime import datetime
from pathlib import Path
from ultralytics import YOLO, YOLOWorld

VAL_RATIO = 0.2
random.seed(42)

DRIVE_ROOT  = '/content/drive/MyDrive/galapagos_wildlife'
IMAGES_DIR  = Path(f'{DRIVE_ROOT}/raw_images')
YOLO_DIR    = Path(f'{DRIVE_ROOT}/yolo')
DATASET_DIR = Path('/content/dataset')

USE_DRIVE = os.path.exists(DRIVE_ROOT)
if not USE_DRIVE:
    print('ERROR: Drive no montado. Ejecuta Celda 0 primero.')
else:
    YOLO_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Drive OK — {DRIVE_ROOT}')

# Leer SPECIES desde species_config.json (fuente unica de verdad)
config_path = Path(f'{DRIVE_ROOT}/species_config.json')
if not config_path.exists():
    raise FileNotFoundError(
        'species_config.json no encontrado.\n'
        'Ejecuta 00_download_dataset.ipynb primero para crear el banco de imagenes.'
    )
config  = json.loads(config_path.read_text())
SPECIES = [(s['id'], s['scientific'], s['en']) for s in config['species']]
print(f'Especies cargadas de Drive: {len(SPECIES)}')

def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

for split in ['train', 'val']:
    (DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

total = sum(
    len(list((IMAGES_DIR / f"{sid:02d}_{sci.replace(' ','_')}").glob('*.jpg')))
    for sid, sci, en in SPECIES
    if (IMAGES_DIR / f"{sid:02d}_{sci.replace(' ','_')}").exists()
)
print(f'Imagenes en banco: {total} ({total // max(1,len(SPECIES))} prom/especie)')
print('Setup completo')


## Celda 3 — Auto-anotación YOLO World (~90-120 min)

In [ ]:
from tqdm import tqdm

log('=== FASE 2: Auto-anotacion con YOLO World ===')
world_model = YOLOWorld('yolov8s-worldv2.pt')
log('YOLO World cargado')

drive_dataset = Path(f'{DRIVE_DIR}/dataset') if USE_DRIVE else None

# ── Mapa de nombres de detección para YOLO World ────────────────────────────
# YOLO World usa nombres genéricos de COCO/WebLI. Para tortugas específicas
# (Chelonoidis spp.) usamos "Galapagos Giant Tortoise" — YOLO lo reconoce.
# La distinción entre especies se aprende durante el entrenamiento YOLO.
YOLO_WORLD_QUERY = {
    'Chelonoidis porteri':      'Galapagos Giant Tortoise',
    'Chelonoidis donfaustoi':   'Galapagos Giant Tortoise',
    'Chelonoidis hoodensis':    'Galapagos Giant Tortoise',
    'Chelonoidis becki':        'Galapagos Giant Tortoise',
    'Chelonoidis vandenburghi': 'Galapagos Giant Tortoise',
    'Chelonoidis darwini':      'Galapagos Giant Tortoise',
}

# ── Pre-scan: determinar clases activas (las que tienen imágenes) ────────────
# Esto nos da los label_ids secuenciales ANTES de anotar,
# para que los archivos .txt y el YAML usen los mismos IDs (0-based).
active_classes = []
for class_id, sci_name, common_name in SPECIES:
    cls_key = f"{class_id:02d}_{sci_name.replace(' ', '_')}"
    cls_dir = IMAGES_DIR / cls_key
    if cls_dir.exists() and list(cls_dir.glob('*.jpg')):
        active_classes.append((class_id, sci_name, common_name))

# Mapa orig_id → label_id secuencial (0-based) para archivos .txt
label_id_map = {orig: seq for seq, (orig, _, _) in enumerate(active_classes)}
n_classes = len(active_classes)
print(f'Clases activas: {n_classes}')
changed = [(orig, seq) for orig, seq in label_id_map.items() if orig != seq]
if changed:
    print(f'IDs remapeados (orig→label): {changed[:5]}{"..." if len(changed)>5 else ""}')

total_det = 0
total_fallback = 0
skipped = 0

for seq_id, (class_id, sci_name, common_name) in enumerate(active_classes):
    cls_key = f"{class_id:02d}_{sci_name.replace(' ', '_')}"
    cls_dir = IMAGES_DIR / cls_key
    imgs = list(cls_dir.glob('*.jpg'))
    prefix = f'[{seq_id+1:02d}/{n_classes}]'

    # ── Restaurar desde Drive si ya fue anotado ───────────────────────────
    if drive_dataset is not None:
        marker = drive_dataset / f'done_{cls_key}.txt'
        if marker.exists():
            restored = 0
            for split in ['train', 'val']:
                for subdir in ['images', 'labels']:
                    src_dir = drive_dataset / subdir / split
                    dst_dir = DATASET_DIR / subdir / split
                    dst_dir.mkdir(parents=True, exist_ok=True)
                    for f in src_dir.glob(f'c{class_id:02d}_*'):
                        target = dst_dir / f.name
                        if not target.exists():
                            shutil.copy(f, target)
                            restored += 1
                        # Remap label files: old orig_id → new seq_id
                        if subdir == 'labels' and target.exists():
                            content = target.read_text().strip()
                            needs_remap = any(
                                line.split()[0] != str(seq_id)
                                for line in content.split('\n') if line.strip()
                            )
                            if needs_remap:
                                new_lines = []
                                for line in content.split('\n'):
                                    parts = line.strip().split()
                                    if parts:
                                        new_lines.append(f'{seq_id} {" ".join(parts[1:])}')
                                target.write_text('\n'.join(new_lines) + '\n')
            skipped += 1
            print(f"{prefix} ✓ {common_name:40s} Drive (seq_id={seq_id}, +{restored} archivos)")
            continue

    # ── Anotar con YOLO World ─────────────────────────────────────────────
    random.shuffle(imgs)
    split_idx = max(1, int(len(imgs) * (1 - VAL_RATIO)))

    detection_name = YOLO_WORLD_QUERY.get(sci_name, common_name)
    world_model.set_classes([detection_name])

    cls_det = 0
    cls_fallback = 0
    det_note = f' (query: "{detection_name}")' if detection_name != common_name else ''
    print(f'{prefix} anotando {common_name}{det_note}: {len(imgs)} imgs  [seq_id={seq_id}]')

    for split_name, split_imgs in [('train', imgs[:split_idx]), ('val', imgs[split_idx:])]:
        out_img = DATASET_DIR / 'images' / split_name
        out_lbl = DATASET_DIR / 'labels' / split_name
        with tqdm(total=len(split_imgs), desc=f'  {split_name:5s} {common_name[:25]}',
                  unit='img', bar_format='{l_bar}{bar:20}{r_bar}') as pbar:
            for img_path in split_imgs:
                stem = f'c{class_id:02d}_{img_path.stem}'
                dst_img = out_img / f'{stem}.jpg'
                dst_lbl = out_lbl / f'{stem}.txt'
                try:
                    res = world_model.predict(str(img_path), conf=0.25, verbose=False)
                    boxes = res[0].boxes
                    if boxes is not None and len(boxes) > 0:
                        best = boxes.conf.argmax().item()
                        x, y, w, h = boxes.xywhn[best].tolist()
                        label = f'{seq_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n'
                        cls_det += 1
                    else:
                        label = f'{seq_id} 0.500000 0.500000 1.000000 1.000000\n'
                        cls_fallback += 1
                    shutil.copy(img_path, dst_img)
                    dst_lbl.write_text(label)
                except Exception:
                    try:
                        shutil.copy(img_path, dst_img)
                        dst_lbl.write_text(f'{seq_id} 0.500000 0.500000 1.000000 1.000000\n')
                        cls_fallback += 1
                    except Exception:
                        pass
                pbar.update(1)

    pct = round(cls_det / max(1, cls_det + cls_fallback) * 100)
    log(f'{prefix} {common_name}: {cls_det} bbox + {cls_fallback} fallback ({pct}% detectado)')
    total_det += cls_det
    total_fallback += cls_fallback

    # Guardar en Drive (checkpoint incremental)
    if drive_dataset is not None:
        drive_dataset.mkdir(parents=True, exist_ok=True)
        for split in ['train', 'val']:
            for subdir in ['images', 'labels']:
                src_dir = DATASET_DIR / subdir / split
                dst_dir = drive_dataset / subdir / split
                dst_dir.mkdir(parents=True, exist_ok=True)
                for f in src_dir.glob(f'c{class_id:02d}_*'):
                    target = dst_dir / f.name
                    if not target.exists():
                        shutil.copy(f, target)
        marker = drive_dataset / f'done_{cls_key}.txt'
        marker.write_text(f'seq_id={seq_id}, {cls_det} bbox, {cls_fallback} fallback')
        log(f'{prefix} guardada en Drive (seq_id={seq_id})')

del world_model
gc.collect()

# ── Generar YAML con IDs secuenciales densos (0-based) ───────────────────────
yaml_lines = ['path: /content/dataset', 'train: images/train', 'val: images/val',
              f'nc: {n_classes}', 'names:']
for seq_id, (orig_id, sci, common) in enumerate(active_classes):
    yaml_lines.append(f'  {seq_id}: {sci}')
yaml_content = '\n'.join(yaml_lines) + '\n'
yaml_path = '/content/dataset/galapagos.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

# Guardar mapa orig_id→seq_id para referencia (e.g. labels.txt en Flutter)
with open('/content/class_id_map.json', 'w') as f:
    json.dump({
        'active_classes': [
            {'orig_id': orig_id, 'seq_id': seq_id, 'scientific': sci, 'common': common}
            for seq_id, (orig_id, sci, common) in enumerate(active_classes)
        ],
        'label_id_map': {str(k): v for k, v in label_id_map.items()},
    }, f, indent=2)

if USE_DRIVE:
    shutil.copy(yaml_path, f'{DRIVE_DIR}/galapagos.yaml')
    shutil.copy('/content/class_id_map.json', f'{DRIVE_DIR}/class_id_map.json')

train_n = len(list((DATASET_DIR / 'images' / 'train').glob('*.jpg')))
val_n   = len(list((DATASET_DIR / 'images' / 'val').glob('*.jpg')))
log(f'=== FASE 2 COMPLETA: {n_classes} clases ({skipped} de Drive) ===')
log(f'  {total_det} bbox detectados, {total_fallback} fallback')
log(f'Dataset: {train_n} train, {val_n} val')

## Celda 4 — Entrenamiento YOLOv8n (~2-3h)

In [ ]:
log('=== FASE 3: Entrenamiento YOLOv8n 100 epochs ===')

last_pt_local = '/content/runs/galapagos_yolo/weights/last.pt'
resume = os.path.exists(last_pt_local)

if resume:
    log('Checkpoint local encontrado — retomando entrenamiento')
    yolo = YOLO(last_pt_local)
    results = yolo.train(resume=True)
else:
    log('Iniciando entrenamiento desde cero')
    yolo = YOLO('yolov8n.pt')
    results = yolo.train(
        data='/content/dataset/galapagos.yaml',
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        save=True,
        save_period=5,
        project='/content/runs',
        name='galapagos_yolo',
        exist_ok=True,
        verbose=True,
    )

map50    = results.results_dict.get('metrics/mAP50(B)', 'N/A')
map50_95 = results.results_dict.get('metrics/mAP50-95(B)', 'N/A')
log(f'Entrenamiento completo — mAP@50: {map50}  mAP@50-95: {map50_95}')

if USE_DRIVE:
    models_dir = YOLO_DIR / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['best.pt', 'last.pt']:
        src = f'/content/runs/galapagos_yolo/weights/{fname}'
        if os.path.exists(src):
            shutil.copy(src, models_dir / fname)
    log_entry = {
        'timestamp': datetime.now().isoformat(),
        'map50': str(map50), 'map50_95': str(map50_95),
        'species_count': len(active_classes),
    }
    log_path = YOLO_DIR / 'training_log.json'
    history = json.loads(log_path.read_text()) if log_path.exists() else []
    history.append(log_entry)
    log_path.write_text(json.dumps(history, indent=2))
    log(f'Modelos guardados en {models_dir}')


## Celda 5 — Export TFLite + Descarga

In [ ]:
log('=== FASE 4: Export TFLite float16 ===')

best_pt = str(YOLO_DIR / 'models' / 'best.pt')
if not os.path.exists(best_pt):
    best_pt = '/content/runs/galapagos_yolo/weights/best.pt'

export_model = YOLO(best_pt)
export_path = export_model.export(
    format='tflite', imgsz=640, half=True, nms=True, simplify=True,
)

labels_path = '/content/galapagos_yolo_labels.txt'
with open(labels_path, 'w') as f:
    for new_id, (orig_id, sci, common) in enumerate(active_classes):
        f.write(f'{sci}|{common}|{common}\n')

tflite_size = os.path.getsize(str(export_path)) / 1024 / 1024
log(f'TFLite: {tflite_size:.1f} MB, {len(active_classes)} clases')

if USE_DRIVE:
    models_dir = YOLO_DIR / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(str(export_path), models_dir / 'galapagos_yolo.tflite')
    shutil.copy(labels_path, models_dir / 'labels.txt')
    log(f'TFLite guardado en {models_dir}')

from google.colab import files
print('\nDescargando para Flutter...')
files.download(labels_path)
files.download(str(export_path))
print('Coloca en assets/ml/ del proyecto Flutter')


## Celda 6 — (Opcional) Ver resultados del entrenamiento

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_img = '/content/runs/galapagos_yolo/results.png'
if os.path.exists(results_img):
    img = mpimg.imread(results_img)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results (loss, mAP, precision, recall)')
    plt.show()
else:
    print('results.png no encontrado - ejecuta la celda de entrenamiento primero')